
# Statistical Analysis Notebook — House Price Dataset

## Important data/report compatibility note

The supplied report is an **e-commerce A/B test** report. It requires variables such as:

- `user_id`
- `timestamp`
- `group` (control/treatment)
- `landing_page` (old/new)
- `converted` (0/1)
- a separate country mapping containing US, UK and CA

The supplied CSV, however, is a **house-price dataset** with 4,600 rows and 18 columns. It contains variables such as `price`, `bedrooms`, `bathrooms`, `sqft_living`, `waterfront`, `city`, etc., and does **not** contain the A/B-test variables required by the report.

Therefore, the notebook does **not fabricate an A/B test**. It first documents the mismatch and then performs the same three statistical techniques from the report in an appropriate house-price setting:

1. Independent two-sample Welch t-test
2. Chi-square test of independence
3. One-way ANOVA

For the adapted analysis:
- **Binary comparison:** waterfront vs. non-waterfront homes, using `price` as the numeric outcome.
- **Chi-square:** waterfront status vs. a binary price category (below/at-or-above the median).
- **ANOVA:** house prices across cities, with an additional city × waterfront analysis where groups have enough observations.

This keeps the statistical methods from the report while using variables that actually exist in the supplied dataset.


## 1. Import libraries and load the supplied dataset

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATA_PATH = Path("House_Price_data.csv")

# If the notebook is run from another folder, change DATA_PATH to the CSV location.
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())


## 2. Data audit and compatibility check

In [ ]:

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nUnique country values:")
print(df["country"].unique())

required_ab_columns = {
    "user_id", "timestamp", "group", "landing_page", "converted"
}
missing_ab_columns = sorted(required_ab_columns - set(df.columns))

print("\nA/B-test columns missing from supplied CSV:")
print(missing_ab_columns)

if missing_ab_columns:
    print(
        "\nCONCLUSION: The supplied CSV cannot reproduce the report's original "
        "e-commerce A/B test exactly. The adapted house-price analysis below "
        "uses the report's statistical methods without inventing missing variables."
    )
else:
    print("All core A/B-test columns are present.")


## 3. Data cleaning

In [ ]:

# Work on a copy
data = df.copy()

# Convert date to datetime
data["date"] = pd.to_datetime(data["date"], errors="coerce")

# Remove exact duplicate rows
before = len(data)
data = data.drop_duplicates().copy()
duplicates_removed = before - len(data)

# Remove records with missing/invalid price
before_price = len(data)
data = data[data["price"].notna() & np.isfinite(data["price"]) & (data["price"] > 0)].copy()
invalid_price_removed = before_price - len(data)

print("Exact duplicate rows removed:", duplicates_removed)
print("Invalid/missing price rows removed:", invalid_price_removed)
print("Rows after cleaning:", len(data))

display(data.describe(include="all").T)


## 4. Descriptive overview

In [ ]:

print("Numerical summary:")
display(data.select_dtypes(include=np.number).describe().T)

print("\nHouse counts by waterfront status:")
display(data["waterfront"].value_counts().rename(index={0: "Non-waterfront", 1: "Waterfront"}))

print("\nTop 15 cities by number of houses:")
display(data["city"].value_counts().head(15).to_frame("house_count"))

print("\nPrice summary:")
display(data["price"].describe())


## 5. Exploratory visualizations

In [ ]:

# Price distribution
plt.figure(figsize=(9, 5))
plt.hist(data["price"], bins=50)
plt.xlabel("House price")
plt.ylabel("Number of houses")
plt.title("Distribution of House Prices")
plt.tight_layout()
plt.show()


In [ ]:

# Price by waterfront status
groups = [
    data.loc[data["waterfront"] == 0, "price"],
    data.loc[data["waterfront"] == 1, "price"]
]

plt.figure(figsize=(8, 5))
plt.boxplot(groups, labels=["Non-waterfront", "Waterfront"], showfliers=False)
plt.ylabel("House price")
plt.title("House Price by Waterfront Status")
plt.tight_layout()
plt.show()


In [ ]:

# Average price for the 15 most represented cities
top_cities = data["city"].value_counts().head(15).index
city_mean = (
    data[data["city"].isin(top_cities)]
    .groupby("city")["price"]
    .mean()
    .sort_values()
)

plt.figure(figsize=(10, 6))
plt.barh(city_mean.index, city_mean.values)
plt.xlabel("Mean house price")
plt.ylabel("City")
plt.title("Mean House Price — 15 Most Represented Cities")
plt.tight_layout()
plt.show()



## 6. Test 1 — Independent two-sample Welch t-test

### Adaptation of the report

The report compares conversion means between control and treatment. Since the supplied dataset has no control/treatment variable, this notebook compares **mean house price between non-waterfront and waterfront properties**.

- **H0:** Mean price of waterfront homes = mean price of non-waterfront homes.
- **H1:** The two mean prices are different.
- Significance level: **α = 0.05**
- Welch's t-test is used because it does not require equal variances.


In [ ]:

non_waterfront = data.loc[data["waterfront"] == 0, "price"].dropna()
waterfront = data.loc[data["waterfront"] == 1, "price"].dropna()

t_stat, p_value = stats.ttest_ind(
    waterfront,
    non_waterfront,
    equal_var=False,
    alternative="two-sided"
)

mean_non = non_waterfront.mean()
mean_water = waterfront.mean()
difference = mean_water - mean_non

# Cohen's d using pooled standard deviation
n1, n0 = len(waterfront), len(non_waterfront)
s1, s0 = waterfront.std(ddof=1), non_waterfront.std(ddof=1)
pooled_sd = np.sqrt(((n1 - 1)*s1**2 + (n0 - 1)*s0**2) / (n1 + n0 - 2))
cohens_d = difference / pooled_sd

print(f"Non-waterfront mean price: {mean_non:,.2f}")
print(f"Waterfront mean price:     {mean_water:,.2f}")
print(f"Difference (waterfront - non-waterfront): {difference:,.2f}")
print(f"Welch t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6g}")
print(f"Cohen's d: {cohens_d:.4f}")

if p_value < 0.05:
    print("Decision: Reject H0 at α = 0.05.")
else:
    print("Decision: Fail to reject H0 at α = 0.05.")


### 95% confidence interval for the mean-price difference

In [ ]:

# Welch-Satterthwaite degrees of freedom
se = np.sqrt(s1**2/n1 + s0**2/n0)
df_welch = (s1**2/n1 + s0**2/n0)**2 / (
    (s1**2/n1)**2/(n1-1) + (s0**2/n0)**2/(n0-1)
)

alpha = 0.05
critical_t = stats.t.ppf(1 - alpha/2, df_welch)
ci_low = difference - critical_t * se
ci_high = difference + critical_t * se

print(f"Welch degrees of freedom: {df_welch:.2f}")
print(f"95% CI for (waterfront - non-waterfront) mean price difference:")
print(f"[{ci_low:,.2f}, {ci_high:,.2f}]")



## 7. Test 2 — Chi-square test of independence

The report uses a 2×2 table of **strategy × conversion**. The supplied house dataset has no conversion variable.

To apply the same categorical method without inventing data, we create a documented binary price category:

- **Below median price**
- **At or above median price**

We then test whether **waterfront status** and this price category are independent.

- **H0:** Waterfront status and price category are independent.
- **H1:** They are associated.
- **α = 0.05**


In [ ]:

median_price = data["price"].median()

data["price_category"] = np.where(
    data["price"] < median_price,
    "Below median",
    "At/above median"
)

contingency = pd.crosstab(
    data["waterfront"].map({0: "Non-waterfront", 1: "Waterfront"}),
    data["price_category"]
)

print("Median house price:", f"{median_price:,.2f}")
display(contingency)

chi2, chi_p, chi_df, expected = stats.chi2_contingency(contingency)

print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {chi_df}")
print(f"p-value: {chi_p:.6g}")

print("\nExpected frequencies:")
display(pd.DataFrame(
    expected,
    index=contingency.index,
    columns=contingency.columns
))

if chi_p < 0.05:
    print("Decision: Reject H0 at α = 0.05.")
else:
    print("Decision: Fail to reject H0 at α = 0.05.")



## 8. Test 3 — One-way ANOVA across cities

The report uses one-way ANOVA to compare conversion rates across countries. The supplied dataset has only one country (`USA`), so country-level ANOVA is not meaningful.

Instead, we use **city** as the geographical grouping variable.

To avoid extremely small groups dominating the test, cities with fewer than 20 observations are excluded from the ANOVA.

- **H0:** All included city mean prices are equal.
- **H1:** At least one city mean price differs.
- **α = 0.05**


In [ ]:

min_city_n = 20

city_counts = data["city"].value_counts()
eligible_cities = city_counts[city_counts >= min_city_n].index

city_groups = [
    data.loc[data["city"] == city, "price"].dropna()
    for city in eligible_cities
]

f_stat, anova_p = stats.f_oneway(*city_groups)

anova_summary = (
    data[data["city"].isin(eligible_cities)]
    .groupby("city")["price"]
    .agg(["count", "mean", "std"])
    .sort_values("mean", ascending=False)
)

display(anova_summary)

print(f"Number of cities included: {len(eligible_cities)}")
print(f"ANOVA F-statistic: {f_stat:.4f}")
print(f"ANOVA p-value: {anova_p:.6g}")

if anova_p < 0.05:
    print("Decision: Reject H0 at α = 0.05.")
    print("At least one included city has a statistically different mean house price.")
else:
    print("Decision: Fail to reject H0 at α = 0.05.")



## 9. Additional ANOVA — City × Waterfront combinations

The report also considers six country × strategy combinations. This dataset has one country, so the closest available structure is **city × waterfront status**.

Only combinations with at least 10 observations are included.


In [ ]:

data["city_waterfront"] = (
    data["city"] + " | " +
    data["waterfront"].map({0: "Non-waterfront", 1: "Waterfront"})
)

combo_counts = data["city_waterfront"].value_counts()
eligible_combos = combo_counts[combo_counts >= 10].index

combo_groups = [
    data.loc[data["city_waterfront"] == combo, "price"].dropna()
    for combo in eligible_combos
]

if len(combo_groups) >= 2:
    f_combo, p_combo = stats.f_oneway(*combo_groups)

    combo_summary = (
        data[data["city_waterfront"].isin(eligible_combos)]
        .groupby("city_waterfront")["price"]
        .agg(["count", "mean", "std"])
        .sort_values("mean", ascending=False)
    )

    display(combo_summary)

    print(f"Number of city × waterfront groups included: {len(eligible_combos)}")
    print(f"ANOVA F-statistic: {f_combo:.4f}")
    print(f"ANOVA p-value: {p_combo:.6g}")

    if p_combo < 0.05:
        print("Decision: Reject H0 at α = 0.05.")
    else:
        print("Decision: Fail to reject H0 at α = 0.05.")
else:
    print("Not enough city × waterfront groups with at least 10 observations to run the ANOVA.")



## 10. Overall results summary

This table summarizes the three statistical methods used in the adapted analysis.


In [ ]:

results = pd.DataFrame({
    "Test": [
        "Welch independent t-test",
        "Chi-square independence",
        "One-way ANOVA — city",
        "One-way ANOVA — city × waterfront"
    ],
    "Null hypothesis": [
        "Mean price is equal for waterfront and non-waterfront homes",
        "Waterfront status and price category are independent",
        "Mean price is equal across included cities",
        "Mean price is equal across included city × waterfront groups"
    ],
    "Statistic": [
        t_stat,
        chi2,
        f_stat,
        f_combo if 'f_combo' in globals() else np.nan
    ],
    "p-value": [
        p_value,
        chi_p,
        anova_p,
        p_combo if 'p_combo' in globals() else np.nan
    ]
})

results["Decision at α=0.05"] = np.where(
    results["p-value"] < 0.05,
    "Reject H0",
    "Fail to reject H0"
)

display(results)



## 11. Interpretation and conclusion

### Important distinction

The supplied report's original conclusion concerns an **e-commerce pricing A/B test**. That conclusion cannot be reproduced from this house-price CSV because the required A/B-test variables are absent.

The results in this notebook therefore answer **house-price questions**, not the original e-commerce question.

Use the p-value with α = 0.05 as follows:

- **p < 0.05:** statistically significant evidence against H0.
- **p ≥ 0.05:** insufficient statistical evidence to reject H0.

A statistically significant result does not by itself establish causation. In particular, the waterfront comparison here is observational, not a randomized A/B experiment.

### Final data limitation

To reproduce the supplied report exactly, the correct e-commerce A/B-test data must contain the fields described in the report (`user id`, `timestamp`, `group`, `landing page`, `converted`) plus the country mapping. The current `House_Price_data.csv` cannot be substituted for that dataset without changing the research question.



## 12. Reproducibility checklist

- [x] Load the supplied CSV
- [x] Inspect data types and missing values
- [x] Check duplicates
- [x] Clean invalid price records
- [x] Descriptive statistics
- [x] Visualize the outcome
- [x] Welch independent two-sample t-test
- [x] 95% confidence interval
- [x] Cohen's d
- [x] Chi-square test of independence
- [x] One-way ANOVA
- [x] Additional group-combination ANOVA
- [x] Report p-values and decisions at α = 0.05
- [x] Explicitly document the mismatch between the supplied report and CSV
